# `train_prod` on Colab GPU

Einfacher Start fuer den ersten GPU-Run: GPU in Colab aktivieren, in der Konfig-Zelle nur den Datensatzpfad anpassen, dann `Run all`.

Fuer den neuen Pfad unten einfach `RUN_TRAIN = run_train_prod_2` setzen.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/miwehle/Translator2.git'
REPO_BRANCH = 'integration/p-to-t'
REPO_DIR = Path('/content/Translator2')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Colab; skipping Drive mount.')

if REPO_DIR.exists():
    print(f'Reusing existing checkout at {REPO_DIR}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH],
    check=True,
)
os.chdir(REPO_DIR)
%pip install -q torch datasets

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Working directory: {Path.cwd()}')
print(f'Git branch: {REPO_BRANCH}')

In [ ]:
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/translator_data/europarl.mapped')
RUN_NAME = 'run1'

TRAIN_KWARGS = {
    'emb_dim': 256,
    'hidden_dim': 1024,
    'num_heads': 8,
    'num_layers': 4,
    'dropout': 0.1,
    'lr': 3e-4,
    'batch_size': 64,
    'epochs': 3,
    'attention': 'torch',
    'max_examples': None,
}

DATASET_PATH

In [ ]:
import torch
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from translator.train_prod import (
    build_train_prod_config,
    run_train_prod,
    run_train_prod_2,
)

RUN_TRAIN = run_train_prod

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}')

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

RUN_DIR = Path('/content/drive/MyDrive/translator_runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

config = build_train_prod_config(
    dataset_path=DATASET_PATH,
    device=device,
    checkpoint_path=RUN_DIR / 'model.pt',
    summary_path=RUN_DIR / 'summary.json',
    **TRAIN_KWARGS,
)

config

In [ ]:
summary = RUN_TRAIN(config)
summary